In [1]:
from collections import defaultdict
from datetime import datetime
import itertools
import json
import os


In [2]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "raw")

intermediate_dir = os.path.join(data_dir, "intermediate")
os.makedirs(intermediate_dir, exist_ok=True)

gameplay_trace = os.path.join("Gameplay", "traces.ndjson")


In [3]:
def load_ndjson(path):
	lines = []

	with open(path, "r", encoding="utf-8") as f:
		for i, line in enumerate(f):
			line = line.strip()
			if line:
				try:
					lines.append(json.loads(line))
				except json.JSONDecodeError:
					print(f"Invalid JSON in {path} (line {i}): {line[:200]}")
	return lines

def save_ndjson(path: str, lines: list[dict]):
	if not path.endswith(".ndjson"):
		path += ".ndjson"

	os.makedirs(os.path.dirname(path), exist_ok=True)

	with open(path, 'w') as f:
		for line in lines:
			json.dump(line, f)
			f.write("\n")

def load_traces(base_dir: str):
	gameplay_dir = os.path.join(base_dir, "Gameplay")

	traces = []
	for file in os.listdir(gameplay_dir):
		if file.endswith(".ndjson"):
			file_path = os.path.join(gameplay_dir, file)
			content = load_ndjson(file_path)
			traces.append(content)

	return list(itertools.chain.from_iterable(traces))


In [4]:
GAME_START_OBJ_ID = "https://w3id.org/xapi/seriousgames/activity-types/serious-game/GameStart"
GAME_END_OBJ_ID = "https://w3id.org/xapi/seriousgames/activity-types/serious-game/GameEnd"


In [5]:
def parse_time(timestamp: str | None):
	if not timestamp:
		return datetime.fromisoformat("1970-01-01T00:00:00+00:00")
	
	return datetime.fromisoformat(timestamp.replace("Z", "+00:00"))

def clean_traces(traces: list):
	traces_sorted = sorted(
		traces,
		key=lambda trace: parse_time(trace.get("timestamp"))
	)

	user_state = defaultdict(lambda: {
		"started": False,
		"ended": False
	})

	first_session_traces = []

	for trace in traces_sorted:
		user = trace.get("actor", {}).get("account", {}).get("name")
		if user:
			obj_id = trace.get("object", {}).get("id", "")
			state = user_state[user]

			if obj_id == GAME_START_OBJ_ID:
				if not state["started"]:
					state["started"] = True
					first_session_traces.append(trace)
					
			elif obj_id == GAME_END_OBJ_ID:
				if state["started"] and not state["ended"]:
					state["ended"] = True
					first_session_traces.append(trace)

			else:
				if state["started"] and not state["ended"]:
					first_session_traces.append(trace)

	valid_users = {
		user for user, state in user_state.items()
		if state["started"] and state["ended"]
	}

	filtered_traces = [
		trace for trace in first_session_traces
		if trace.get("actor", {}).get("account", {}).get("name") in valid_users
	]

	return filtered_traces, valid_users


In [6]:
common_users = None
merged_traces = []

for session_name in os.listdir(raw_dir):
	session_dir = os.path.join(raw_dir, session_name)

	print(f"\nSession: {session_name}")

	traces = load_traces(session_dir)
	filtered_traces, valid_users = clean_traces(traces)

	merged_traces.append(filtered_traces)

	print("Users who completed first session:", len(valid_users))
	print("Total traces:", len(traces))
	print("Filtered traces:", len(filtered_traces))

	if common_users is None:
		common_users = valid_users
	else:
		common_users &= valid_users
		
	output_path = os.path.join(
		intermediate_dir,
		session_name,
		gameplay_trace
	)
	
	save_ndjson(output_path, filtered_traces)

merged_traces = list(itertools.chain.from_iterable(merged_traces))

print("\nSame users across sessions:", len(common_users))
print("Total merged traces:", len(merged_traces))

output_path = os.path.join(
	intermediate_dir,
	"merged",
	gameplay_trace
)

save_ndjson(output_path, merged_traces)



Session: CarpeDiem-11-06-2025
Users who completed first session: 32
Total traces: 34186
Filtered traces: 30095

Session: CarpeDiem-20-05-2025
Users who completed first session: 49
Total traces: 48456
Filtered traces: 45913

Session: CarpeDiem-26-05-2026
Users who completed first session: 35
Total traces: 37329
Filtered traces: 34395

Same users across sessions: 0
Total merged traces: 110403
